In [ ]:
# ============================================================
# Vision2Drive Training Notebook
# Part 1 - Imports
# ============================================================

# Standard Library
import os
import time
import random
from pathlib import Path

# Numerical Computing
import numpy as np

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim

# Data Loading
from torch.utils.data import DataLoader

# Visualization
import matplotlib.pyplot as plt

# Progress Bar
from tqdm.auto import tqdm

# Metrics
from sklearn.metrics import mean_absolute_error

# ------------------------------------------------------------
# Project Imports
# ------------------------------------------------------------

from dataset.dataset import Vision2DriveDataset
from models.model import Vision2Drive

print("=" * 60)
print("✅ All libraries imported successfully.")
print("=" * 60)

In [ ]:
# ============================================================
# Part 2 - Configuration
# ============================================================

CONFIG = {

    # --------------------------
    # Dataset
    # --------------------------
    "dataset_root": "dataset",

    "train_batch_size": 16,
    "val_batch_size": 16,

    "num_workers": 4,

    # --------------------------
    # Model
    # --------------------------
    "image_size": 224,

    "lidar_channels": 5,

    # --------------------------
    # Training
    # --------------------------
    "epochs": 50,

    "learning_rate": 1e-4,

    "weight_decay": 1e-5,

    # --------------------------
    # Scheduler
    # --------------------------
    "scheduler_step": 10,

    "scheduler_gamma": 0.5,

    # --------------------------
    # Checkpoints
    # --------------------------
    "checkpoint_dir": "checkpoints",

    "save_best_only": True,

    # --------------------------
    # Random Seed
    # --------------------------
    "seed": 42,
}

print("=" * 60)
print("Training Configuration")
print("=" * 60)

for key, value in CONFIG.items():
    print(f"{key:<25}: {value}")

In [ ]:
# ============================================================
# Part 3 - Reproducibility & Device Setup
# ============================================================

def set_seed(seed: int = 42):
    """
    Sets all random seeds for reproducible experiments.
    """

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.mps.is_available():
        torch.manual_seed(seed)

    # Reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# Set Random Seed
# ------------------------------------------------------------

set_seed(CONFIG["seed"])


# ------------------------------------------------------------
# Select Device
# ------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")

elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")

else:
    DEVICE = torch.device("cpu")


print("=" * 60)
print("Experiment Setup")
print("=" * 60)

print(f"Random Seed : {CONFIG['seed']}")
print(f"Device      : {DEVICE}")

if DEVICE.type == "cuda":
    print(f"GPU         : {torch.cuda.get_device_name(0)}")

print("=" * 60)

In [ ]:
# ============================================================
# Part 4 - Dataset Initialization
# ============================================================

print("=" * 60)
print("Loading Vision2Drive Dataset")
print("=" * 60)


train_dataset = Vision2DriveDataset(
    root_dir=CONFIG["dataset_root"],
    split="train"
)

val_dataset = Vision2DriveDataset(
    root_dir=CONFIG["dataset_root"],
    split="val"
)

test_dataset = Vision2DriveDataset(
    root_dir=CONFIG["dataset_root"],
    split="test"
)


print(f"Training Samples   : {len(train_dataset)}")
print(f"Validation Samples : {len(val_dataset)}")
print(f"Test Samples       : {len(test_dataset)}")

print("=" * 60)

In [ ]:
# ============================================================
# Part 5 - DataLoader
# ============================================================

print("=" * 60)
print("Creating DataLoaders")
print("=" * 60)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["train_batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["val_batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["val_batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=(DEVICE.type == "cuda"),
)

print(f"Training Batches   : {len(train_loader)}")
print(f"Validation Batches : {len(val_loader)}")
print(f"Test Batches       : {len(test_loader)}")

print("=" * 60)

In [ ]:
# ============================================================
# Part 6 - Dataset Visualization
# ============================================================

sample = train_dataset[0]

print("=" * 60)
print("Dataset Sample Inspection")
print("=" * 60)

print(f"Available Keys : {list(sample.keys())}")

print()

for key, value in sample.items():

    if torch.is_tensor(value):
        print(f"{key:<15} Shape: {tuple(value.shape)}")
    else:
        print(f"{key:<15} Value: {value}")

print("=" * 60)



In [ ]:
# ============================================================
# RGB Image & LiDAR Visualization
# ============================================================

rgb = sample["image"].permute(1, 2, 0).numpy()

rgb = (rgb * 0.5) + 0.5
rgb = np.clip(rgb, 0, 1)

lidar = sample["lidar"][0].numpy()

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].imshow(rgb)
ax[0].set_title("RGB Camera")
ax[0].axis("off")

ax[1].imshow(lidar, cmap="gray")
ax[1].set_title("LiDAR BEV (Channel 0)")
ax[1].axis("off")

plt.show()

In [ ]:
print("=" * 60)
print("Vehicle State")
print("=" * 60)

print(sample["state"])


print("=" * 60)
print("Navigation")
print("=" * 60)

print(sample["navigation"])

print("=" * 60)
print("Expert Driving Command")
print("=" * 60)

print(f"Steering : {sample['action'][0]:.3f}")
print(f"Throttle : {sample['action'][1]:.3f}")
print(f"Brake    : {sample['action'][2]:.3f}")

In [ ]:
# ============================================================
# Part 7 - Build Vision2Drive Model
# ============================================================

print("=" * 60)
print("Building Vision2Drive Model")
print("=" * 60)

# ------------------------------------------------------------
# Create Model
# ------------------------------------------------------------

model = Vision2Drive()

# ------------------------------------------------------------
# Move Model to Device
# ------------------------------------------------------------

model = model.to(DEVICE)

# ------------------------------------------------------------
# Training Mode
# ------------------------------------------------------------

model.train()

print(f"Device : {DEVICE}")
print(f"Model  : {model.__class__.__name__}")

print("=" * 60)
print("✅ Vision2Drive successfully initialized.")
print("=" * 60)

In [ ]:
# ============================================================
# Model Statistics
# ============================================================

total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("Model Statistics")
print("=" * 60)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

print("=" * 60)

In [ ]:
# ============================================================
# Load One Batch
# ============================================================

batch = next(iter(train_loader))

images = batch["image"].to(DEVICE)
lidar = batch["lidar"].to(DEVICE)
state = batch["state"].to(DEVICE)
navigation = batch["navigation"].to(DEVICE)

In [ ]:
# ============================================================
# Forward Pass
# ============================================================

with torch.no_grad():

    outputs = model(
        images,
        lidar,
        state,
        navigation
    )

In [ ]:
# ============================================================
# Output Verification
# ============================================================

print("=" * 60)
print("Forward Pass Verification")
print("=" * 60)

print(f"Input Image Shape      : {tuple(images.shape)}")
print(f"Input LiDAR Shape      : {tuple(lidar.shape)}")
print(f"Vehicle State Shape    : {tuple(state.shape)}")
print(f"Navigation Shape       : {tuple(navigation.shape)}")

print()

print(f"Steering Shape         : {tuple(outputs['steering'].shape)}")
print(f"Throttle Shape         : {tuple(outputs['throttle'].shape)}")
print(f"Brake Shape            : {tuple(outputs['brake'].shape)}")

print("=" * 60)


# ============================================================
# Prediction Statistics
# ============================================================

print("=" * 60)
print("Prediction Statistics")
print("=" * 60)

for key, value in outputs.items():

    print(f"{key:<12}")

    print(f"Mean : {value.mean().item():.4f}")

    print(f"Std  : {value.std().item():.4f}")

    print(f"Min  : {value.min().item():.4f}")

    print(f"Max  : {value.max().item():.4f}")

    print()

In [ ]:
#part 9

# ============================================================
# Part 9 - Driving Loss Function
# ============================================================

print("=" * 60)
print("Building Driving Loss")
print("=" * 60)

criterion = nn.MSELoss()

print("Loss Function :", criterion)

print("=" * 60)

: 

In [ ]:
# ============================================================
# Compute Driving Loss
# ============================================================

ground_truth = batch["action"].to(DEVICE)

steer_gt = ground_truth[:, 0].unsqueeze(1)
throttle_gt = ground_truth[:, 1].unsqueeze(1)
brake_gt = ground_truth[:, 2].unsqueeze(1)

steering_loss = criterion(
    outputs["steering"],
    steer_gt
)

throttle_loss = criterion(
    outputs["throttle"],
    throttle_gt
)

brake_loss = criterion(
    outputs["brake"],
    brake_gt)

total_loss = (
    steering_loss +
    throttle_loss +
    brake_loss
)
# ============================================================
# Loss Statistics
# ============================================================

print("=" * 60)
print("Driving Loss")
print("=" * 60)

print(f"Steering Loss : {steering_loss.item():.6f}")
print(f"Throttle Loss : {throttle_loss.item():.6f}")
print(f"Brake Loss    : {brake_loss.item():.6f}")

print("-" * 60)

print(f"Total Loss    : {total_loss.item():.6f}")

print("=" * 60)

In [ ]:
# ============================================================
# Part 10 - Optimizer
# ============================================================

print("=" * 60)
print("Building Optimizer")
print("=" * 60)

optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

print(optimizer)

print("=" * 60)

# ============================================================
# Optimizer Information
# ============================================================

print("=" * 60)
print("Optimizer Configuration")
print("=" * 60)

print(f"Learning Rate : {CONFIG['learning_rate']}")

print(f"Weight Decay  : {CONFIG['weight_decay']}")

print(f"Optimizer     : AdamW")

print("=" * 60)

In [ ]:
# ============================================================
# Part 11 - Learning Rate Scheduler
# ============================================================

print("=" * 60)
print("Building Learning Rate Scheduler")
print("=" * 60)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=CONFIG["scheduler_step"],
    gamma=CONFIG["scheduler_gamma"]
)

print("Scheduler :", scheduler.__class__.__name__)

print("=" * 60)

# ============================================================
# Current Learning Rate
# ============================================================

current_lr = optimizer.param_groups[0]["lr"]

print("=" * 60)
print("Learning Rate")
print("=" * 60)

print(f"Current LR : {current_lr:.6f}")

print("=" * 60)

In [ ]:
# ============================================================
# Simulated Learning Rate Schedule
# ============================================================

epochs = list(range(CONFIG["epochs"]))

lr = CONFIG["learning_rate"]

lr_values = []

for epoch in epochs:

    lr_values.append(lr)

    if (epoch + 1) % CONFIG["scheduler_step"] == 0:
        lr *= CONFIG["scheduler_gamma"]

plt.figure(figsize=(8,4))

plt.plot(
    epochs,
    lr_values,
    linewidth=2
)

plt.title("Learning Rate Schedule")

plt.xlabel("Epoch")

plt.ylabel("Learning Rate")

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# Part 12 - Training Metrics
# ============================================================

history = {

    "train_loss": [],

    "val_loss": [],

    "steering_loss": [],

    "throttle_loss": [],

    "brake_loss": [],

    "learning_rate": []

}

print("=" * 60)
print("Training History Initialized")
print("=" * 60)

for key in history.keys():

    print("✓", key)

print("=" * 60)

# ============================================================
# Metric Logger
# ============================================================

def log_metrics(

    history,

    train_loss,

    val_loss,

    steering,

    throttle,

    brake,

    lr

):

    history["train_loss"].append(train_loss)

    history["val_loss"].append(val_loss)

    history["steering_loss"].append(steering)

    history["throttle_loss"].append(throttle)

    history["brake_loss"].append(brake)

    history["learning_rate"].append(lr)

In [ ]:
# ============================================================
# Current Training Status
# ============================================================

print("=" * 60)
print("Metric Dashboard")
print("=" * 60)

print(f"Train Loss : {len(history['train_loss'])}")

print(f"Val Loss   : {len(history['val_loss'])}")

print(f"LR History : {len(history['learning_rate'])}")

print("=" * 60)

In [ ]:
# ============================================================
# Part 13 - Training Function
# ============================================================

def train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    device
):
    """
    Trains Vision2Drive for one epoch.
    """

    model.train()

    running_loss = 0.0

    steering_running = 0.0
    throttle_running = 0.0
    brake_running = 0.0

    progress_bar = tqdm(
        train_loader,
        desc="Training",
        leave=False
    )

    for batch in progress_bar:

        images = batch["image"].to(device)
        lidar = batch["lidar"].to(device)
        state = batch["state"].to(device)
        navigation = batch["navigation"].to(device)

        action = batch["action"].to(device)

        steer_gt = action[:, 0].unsqueeze(1)
        throttle_gt = action[:, 1].unsqueeze(1)
        brake_gt = action[:, 2].unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(
            images,
            lidar,
            state,
            navigation
        )

        steering_loss = criterion(
            outputs["steering"],
            steer_gt
        )

        throttle_loss = criterion(
            outputs["throttle"],
            throttle_gt
        )

        brake_loss = criterion(
            outputs["brake"],
            brake_gt
        )

        loss = (
            steering_loss +
            throttle_loss +
            brake_loss
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        steering_running += steering_loss.item()
        throttle_running += throttle_loss.item()
        brake_running += brake_loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    num_batches = len(train_loader)

    return {

        "loss": running_loss / num_batches,

        "steering": steering_running / num_batches,

        "throttle": throttle_running / num_batches,

        "brake": brake_running / num_batches

    }

In [ ]:
# ============================================================
# Part 14 - Validation Function
# ============================================================

def validate(
    model,
    val_loader,
    criterion,
    device
):
    """
    Evaluates Vision2Drive on the validation set.
    """

    model.eval()

    running_loss = 0.0

    steering_running = 0.0
    throttle_running = 0.0
    brake_running = 0.0

    with torch.no_grad():

        progress_bar = tqdm(
            val_loader,
            desc="Validation",
            leave=False
        )

        for batch in progress_bar:

            images = batch["image"].to(device)
            lidar = batch["lidar"].to(device)
            state = batch["state"].to(device)
            navigation = batch["navigation"].to(device)

            action = batch["action"].to(device)

            steer_gt = action[:, 0].unsqueeze(1)
            throttle_gt = action[:, 1].unsqueeze(1)
            brake_gt = action[:, 2].unsqueeze(1)

            outputs = model(
                images,
                lidar,
                state,
                navigation
            )

            steering_loss = criterion(
                outputs["steering"],
                steer_gt
            )

            throttle_loss = criterion(
                outputs["throttle"],
                throttle_gt
            )

            brake_loss = criterion(
                outputs["brake"],
                brake_gt
            )

            loss = (
                steering_loss +
                throttle_loss +
                brake_loss
            )

            running_loss += loss.item()

            steering_running += steering_loss.item()
            throttle_running += throttle_loss.item()
            brake_running += brake_loss.item()

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

    num_batches = len(val_loader)

    return {

        "loss": running_loss / num_batches,

        "steering": steering_running / num_batches,

        "throttle": throttle_running / num_batches,

        "brake": brake_running / num_batches

    }

In [ ]:
# ============================================================
# Part 15 - Checkpoint Saving
# ============================================================

print("=" * 60)
print("Checkpoint Configuration")
print("=" * 60)

os.makedirs(
    CONFIG["checkpoint_dir"],
    exist_ok=True
)

print(f"Directory : {CONFIG['checkpoint_dir']}")

print("=" * 60)

In [ ]:
# ============================================================
# Save Checkpoint
# ============================================================

def save_checkpoint(
    model,
    optimizer,
    scheduler,
    epoch,
    val_loss,
    path
):
    checkpoint = {

        "epoch": epoch,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "scheduler_state_dict": scheduler.state_dict(),

        "val_loss": val_loss

    }

    torch.save(
        checkpoint,
        path
    )

In [ ]:
# ============================================================
# Part 16 - Complete Training Loop
# ============================================================

print("=" * 60)
print("Starting Vision2Drive Training")
print("=" * 60)

best_val_loss = float("inf")

start_time = time.time()

for epoch in range(CONFIG["epochs"]):

    print()

    print("-" * 60)

    print(
        f"Epoch {epoch + 1}/{CONFIG['epochs']}"
    )

    print("-" * 60)

    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        DEVICE
    )

    val_metrics = validate(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    log_metrics(

        history,

        train_metrics["loss"],

        val_metrics["loss"],

        train_metrics["steering"],

        train_metrics["throttle"],

        train_metrics["brake"],

        current_lr

    )

    print(
        f"Train Loss : {train_metrics['loss']:.6f}"
    )

    print(
        f"Validation Loss : {val_metrics['loss']:.6f}"
    )

    print(
        f"Learning Rate : {current_lr:.6f}"
    )

    if val_metrics["loss"] < best_val_loss:

        best_val_loss = val_metrics["loss"]

        checkpoint_path = os.path.join(
            CONFIG["checkpoint_dir"],
            "best_model.pth"
        )

        save_checkpoint(
            model,
            optimizer,
            scheduler,
            epoch + 1,
            best_val_loss,
            checkpoint_path
        )

        print("✅ Best model saved!")

print()

total_time = time.time() - start_time

print("=" * 60)

print("Training Finished")

print(f"Best Validation Loss : {best_val_loss:.6f}")

print(f"Training Time : {total_time/60:.2f} minutes")

print("=" * 60)

In [ ]:
# ============================================================
# Part 17 - Training Results Visualization
# ============================================================

epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(10,5))

plt.plot(
    epochs,
    history["train_loss"],
    label="Training Loss",
    linewidth=2
)

plt.plot(
    epochs,
    history["val_loss"],
    label="Validation Loss",
    linewidth=2
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    epochs,
    history["steering_loss"],
    linewidth=2
)

plt.title("Steering Loss")

plt.xlabel("Epoch")

plt.ylabel("MSE")

plt.grid(True)

plt.show()



In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    epochs,
    history["throttle_loss"],
    linewidth=2
)

plt.title("Throttle Loss")

plt.xlabel("Epoch")

plt.ylabel("MSE")

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    epochs,
    history["brake_loss"],
    linewidth=2
)

plt.title("Brake Loss")

plt.xlabel("Epoch")

plt.ylabel("MSE")

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    epochs,
    history["learning_rate"],
    linewidth=2
)

plt.title("Learning Rate")

plt.xlabel("Epoch")

plt.ylabel("LR")

plt.grid(True)

plt.show()

In [ ]:
print("=" * 60)
print("Training Summary")
print("=" * 60)

print(f"Epochs Completed : {len(history['train_loss'])}")

print(f"Final Train Loss : {history['train_loss'][-1]:.6f}")

print(f"Final Val Loss   : {history['val_loss'][-1]:.6f}")

print(f"Best Val Loss    : {min(history['val_loss']):.6f}")

print("=" * 60)

In [ ]:
# ============================================================
# Part 18 - Prediction Visualization
# ============================================================

model.eval()

batch = next(iter(test_loader))

images = batch["image"].to(DEVICE)

lidar = batch["lidar"].to(DEVICE)

state = batch["state"].to(DEVICE)

navigation = batch["navigation"].to(DEVICE)

ground_truth = batch["action"]

In [ ]:
with torch.no_grad():

    predictions = model(
        images,
        lidar,
        state,
        navigation
    )


num_examples = min(5, images.shape[0])

print("=" * 60)
print("Prediction Comparison")
print("=" * 60)

for i in range(num_examples):

    print(f"\nSample {i+1}")

    print("-" * 40)

    print(
        f"Steering  | GT: {ground_truth[i,0]: .3f}"
        f" | Pred: {predictions['steering'][i].item(): .3f}"
    )

    print(
        f"Throttle  | GT: {ground_truth[i,1]: .3f}"
        f" | Pred: {predictions['throttle'][i].item(): .3f}"
    )

    print(
        f"Brake     | GT: {ground_truth[i,2]: .3f}"
        f" | Pred: {predictions['brake'][i].item(): .3f}"
    )

In [ ]:
rgb = images[0].cpu().permute(1,2,0).numpy()

rgb = rgb * 0.5 + 0.5

rgb = np.clip(rgb,0,1)

plt.figure(figsize=(6,6))

plt.imshow(rgb)

plt.title("Input Camera Frame")

plt.axis("off")

plt.show()

In [ ]:
print("=" * 60)
print("Predicted Driving Command")
print("=" * 60)

print(
    f"Steering : {predictions['steering'][0].item():.3f}"
)

print(
    f"Throttle : {predictions['throttle'][0].item():.3f}"
)

print(
    f"Brake    : {predictions['brake'][0].item():.3f}"
)

print("=" * 60)

In [ ]:
# ============================================================
# Part 19 - Load Best Model
# ============================================================

print("=" * 60)
print("Loading Best Vision2Drive Model")
print("=" * 60)

checkpoint_path = os.path.join(
    CONFIG["checkpoint_dir"],
    "best_model.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(f"Loaded Epoch     : {checkpoint['epoch']}")
print(f"Validation Loss  : {checkpoint['val_loss']:.6f}")

print("=" * 60)

In [ ]:
# ============================================================
# Inference
# ============================================================

batch = next(iter(test_loader))

images = batch["image"].to(DEVICE)
lidar = batch["lidar"].to(DEVICE)
state = batch["state"].to(DEVICE)
navigation = batch["navigation"].to(DEVICE)

with torch.no_grad():

    predictions = model(
        images,
        lidar,
        state,
        navigation
    )

In [ ]:
rgb = images[0].cpu().permute(1,2,0).numpy()

rgb = rgb * 0.5 + 0.5
rgb = np.clip(rgb,0,1)

plt.figure(figsize=(7,7))

plt.imshow(rgb)

plt.axis("off")

plt.title("MetaDrive Camera View")

plt.show()

print("=" * 60)
print("Vision2Drive Driving Decision")
print("=" * 60)

print(f"Steering : {predictions['steering'][0].item():.3f}")

print(f"Throttle : {predictions['throttle'][0].item():.3f}")

print(f"Brake    : {predictions['brake'][0].item():.3f}")

print("=" * 60)

In [ ]:
ground_truth = batch["action"]

print("=" * 60)
print("Expert vs Vision2Drive")
print("=" * 60)

print(f"Expert Steering : {ground_truth[0,0]:.3f}")
print(f"Model Steering  : {predictions['steering'][0].item():.3f}")

print()

print(f"Expert Throttle : {ground_truth[0,1]:.3f}")
print(f"Model Throttle  : {predictions['throttle'][0].item():.3f}")

print()

print(f"Expert Brake    : {ground_truth[0,2]:.3f}")
print(f"Model Brake     : {predictions['brake'][0].item():.3f}")

print("=" * 60)

In [ ]:
# ============================================================
# Part 20 - Final Evaluation
# ============================================================

print("=" * 60)
print("Vision2Drive Training Report")
print("=" * 60)

print(f"Epochs Completed : {len(history['train_loss'])}")

print(f"Best Validation Loss : {min(history['val_loss']):.6f}")

print(f"Final Train Loss : {history['train_loss'][-1]:.6f}")

print(f"Final Validation Loss : {history['val_loss'][-1]:.6f}")

print()

print("Model Status : Trained")

print("Framework : PyTorch")

print("Simulator : MetaDrive")

print("=" * 60)

In [ ]:
print("=" * 60)
print("Vision2Drive Complete")
print("=" * 60)

print("✓ Dataset Built")

print("✓ DataLoader Ready")

print("✓ Vision2Drive Constructed")

print("✓ Loss Function Implemented")

print("✓ Optimizer Configured")

print("✓ Scheduler Configured")

print("✓ Model Trained")

print("✓ Validation Completed")

print("✓ Best Checkpoint Saved")

print("✓ MetaDrive Inference Verified")

print("✓ Final Model Exported")

print("=" * 60)